In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_1.silver;

In [0]:
from datetime import datetime

# Задаем минимальную и текущую (максимальную) дату для чистки от неправильных данных по дате
MIN_DATE = '2025-01-01'
MAX_DATE = datetime.now().strftime('%Y-%m-%d')

BRONZE_TABLE = "pipeline_1.bronze.raw_events"
SILVER_TABLE = "pipeline_1.silver.silver_events"
CHECKPOINT_PATH = "/Volumes/pipeline_1/bronze/checkpoints_volume/silver_checkpoint"

In [0]:
from pyspark.sql.functions import explode, col, current_timestamp, substring

df_bronze = spark.readStream.table(BRONZE_TABLE)

# Разворачиваем массив из метрик, чтобы каждая метрика стояла отдельно
# Делаем конвертацию timestamp в корректный формат
# Фильтруем по датам, чтобы отсечь некорректные

df_exploded = (
    df_bronze
    .select(explode(col("values")).alias("metric_row"))
    .select(
        col("metric_row.device_id").alias("device_id"),
        col("metric_row.metric").alias("metric"),
        col("metric_row.value").alias("value"),
        (substring(col("metric_row.timestampUtc").cast("string"), 1, 13).cast("long") / 1000).cast("timestamp").alias("event_time")
    )
    .filter(f"event_time > '{MIN_DATE}' AND event_time < '{MAX_DATE}'")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

def write_to_silver(microBatchDF, batchId):
    # Pivot без хардкода метрик
    pivoted_df = (
        microBatchDF
        .groupBy("device_id", "event_time")
        .pivot("metric")
        .agg({"value": "first"})
    )

    # Маппинг типов для метрик
    type_mapping = {
        "Wind_speed": "float",
        "AirPres": "float",
        "Ambient_temperature": "float",
        "Humidity": "float",
        "Power_output": "float",
        "Rotor_rpm": "int",
        "Error_code": "int",
        "Brake_active": "boolean",
        "Grid_connected": "boolean",
        "Operation_mode": "string"
    }

    # Динамически собираем select и приводим к нужному типу
    select_exprs = []
    for c in pivoted_df.columns:
        if c in type_mapping:
            select_exprs.append(col(c).cast(type_mapping[c]).alias(c))
        else:
            select_exprs.append(col(c))

    result_df = (
        pivoted_df
        .select(*select_exprs)
        .withColumn("inserted_at", current_timestamp())
    )

    (
        result_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE)
    )

(df_exploded
    .writeStream
    .foreachBatch(write_to_silver)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start())